# Woche 4 · Tokens und Embeddings: Wie wird Text zu Zahlen?
**AI Engineering: von LLMs zu Agenten** · linkit e.V. am KIT

Ein neuronales Netz rechnet nur mit Zahlen (Woche 3). Heute klären wir, wie aus Text Zahlen werden, in zwei Schritten:
1. **Tokenisierung:** Text wird in Stücke (Tokens) zerlegt, jedes Token bekommt eine Nummer.
2. **Embeddings:** Jedes Token, und später jeder ganze Satz, wird zu einem Vektor, dessen Richtung Bedeutung trägt.

**Lernziele**
- Tokenisierung per Byte-Pair-Encoding (BPE) erklären und selbst implementieren
- Tokenzahlen messen und die Folgen für Kosten und Kontextlänge benennen
- Embeddings als Vektoren deuten und semantische Ähnlichkeit per Kosinus messen
- Hochdimensionale Vektoren mit PCA auf zwei Dimensionen projizieren

**Aufbau:** Teil A Tokenisierung · Teil B Embedding-Matrix · Teil C Satz-Embeddings und semantische Suche · Teil D PCA
Für TODO 4 braucht ihr euren Toolbox-Schlüssel als Colab-Secret `KIT_TOOLBOX_KEY`. Pflicht: TODO 1–7, Ziel: TODO 8.

In [ ]:
# Setup: Bibliotheken in fester Version (dauert ca. 30 Sekunden). Transformers und PyTorch sind in Colab vorinstalliert.
%pip install -q sentence-transformers==6.1.0 openai==3.24.0

In [ ]:
import math
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import transformers, sentence_transformers, openai
from openai import OpenAI
from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer

print("transformers", transformers.__version__, "| sentence-transformers", sentence_transformers.__version__, "| openai", openai.__version__)

def check(name, test):
    """Selbsttest: zeigt ✅ oder ❌. Fehler im Test gelten als 'noch nicht gelöst'."""
    try:
        ok = bool(test())
    except Exception as e:
        print(f"❌ {name}: noch nicht gelöst ({type(e).__name__}: {e})")
        return
    print(f"✅ {name}" if ok else f"❌ {name}: Ergebnis stimmt noch nicht")

---
# Teil A · Tokenisierung

## A1 · Wie zerlegt GPT-2 unsere Sätze?

Wir laden den **Tokenizer** von GPT-2 (nur den Tokenizer, nicht das Modell). Er hat einen Wortschatz von 50.257 Tokens und wurde überwiegend auf englischen Texten trainiert.
`tok.tokenize(text)` zeigt die Stücke, `tok.encode(text)` die Token-IDs.

In [ ]:
tok = AutoTokenizer.from_pretrained("openai-community/gpt2")
print("Wortschatz:", len(tok), "Tokens")

satz_en = "The spindle of the lathe is running rough and needs a new bearing."
satz_de = "Die Spindel der Drehmaschine läuft unrund und braucht ein neues Lager."

for satz in [satz_en, satz_de]:
    ids = tok.encode(satz)
    stuecke = [tok.decode([i]) for i in ids]
    print(f"\n{len(ids)} Tokens: {satz}")
    print(" | ".join(stuecke))

**Beobachtungen:** Ein Leerzeichen gehört bei GPT-2 zum folgenden Token. Umlaute werden manchmal in zwei Tokens zerlegt, weil GPT-2 auf **Bytes** arbeitet und `ü` in UTF-8 aus zwei Bytes besteht. Einzelne Stücke davon lassen sich nicht als Zeichen anzeigen (`�`).

### TODO 1 · Deutsch gegen Englisch
Berechnet `anzahl_en` und `anzahl_de` (Anzahl Tokens der beiden Sätze) und das `verhaeltnis` Deutsch zu Englisch.

In [ ]:
# TODO 1
anzahl_en = None
anzahl_de = None
verhaeltnis = None

print(anzahl_en, anzahl_de, verhaeltnis)

In [ ]:
check("TODO 1a Anzahl Englisch", lambda: anzahl_en == len(tok.encode(satz_en)))
check("TODO 1b Anzahl Deutsch", lambda: anzahl_de == len(tok.encode(satz_de)))
check("TODO 1c Verhältnis", lambda: math.isclose(verhaeltnis, anzahl_de / anzahl_en))

Und Fachbegriffe? Seltene, lange Wörter zerfallen in viele Stücke.

In [ ]:
for wort in ["bearing", "Lager", "Spindelüberhitzung", "Kühlschmierstoff", "Instandhaltung"]:
    stuecke = tok.tokenize(" " + wort)
    print(f"{wort:20s} {len(stuecke):2d} Tokens: {[tok.decode([i]) for i in tok.encode(' ' + wort)]}")

> **Warum zählt das?** Abgerechnet wird pro Token (Woche 2), und das Kontextfenster eines Modells ist in Tokens begrenzt. Mehr Tokens für denselben Inhalt heißt höhere Kosten und weniger Platz.

## A2 · Byte-Pair-Encoding selbst bauen

Wie entsteht so ein Wortschatz? **BPE** startet mit einzelnen Zeichen und verschmilzt immer wieder das **häufigste benachbarte Paar** zu einem neuen Token.
Unser Mini-Korpus: Wörter als Tupel von Zeichen, mit ihrer Häufigkeit.

In [ ]:
KORPUS = {
    tuple("lager"): 10,
    tuple("lagerschaden"): 4,
    tuple("schaden"): 6,
    tuple("spindel"): 5,
    tuple("spindellager"): 3,
    tuple("kugellager"): 4,
}
print(KORPUS)

### TODO 2 · Paare zählen
`zaehle_paare(korpus)` soll einen `Counter` zurückgeben, der für jedes benachbarte Paar `(a, b)` zählt, wie oft es vorkommt, **gewichtet mit der Worthäufigkeit**.
Beispiel: `("g", "e")` kommt in „lager“ (10-mal), „lagerschaden“ (4), „spindellager“ (3), „kugellager“ (4 Wörter mit je zweimal „ge“) vor.

Tipp: `zip(wort, wort[1:])` liefert alle benachbarten Paare eines Worts.

In [ ]:
def zaehle_paare(korpus):
    zaehler = Counter()
    # TODO 2
    return zaehler

In [ ]:
paare = zaehle_paare(KORPUS)
print(paare.most_common(5))
check("TODO 2a ('g','e') → 25", lambda: paare[("g", "e")] == 25)
check("TODO 2b ('l','a') → 21", lambda: paare[("l", "a")] == 21)
check("TODO 2c ('s','p') → 8", lambda: paare[("s", "p")] == 8)

### TODO 3 · Ein Paar verschmelzen
`merge(korpus, paar)` gibt ein **neues** Korpus zurück, in dem jedes Vorkommen von `paar` (von links nach rechts, ohne Überlappung) durch ein einziges Token ersetzt ist.
Beispiel: Aus `("l","a","g","e","r")` wird mit dem Paar `("g","e")` das Tupel `("l","a","ge","r")`.

In [ ]:
def merge(korpus, paar):
    neu = {}
    for wort, anzahl in korpus.items():
        teile = []
        i = 0
        # TODO 3: wort von links nach rechts durchgehen; passt (wort[i], wort[i+1]) zum Paar,
        #         verschmolzenes Token anhängen und i um 2 erhöhen, sonst wort[i] anhängen und i um 1 erhöhen
        neu[tuple(teile)] = anzahl
    return neu

In [ ]:
check("TODO 3a einfacher Fall", lambda: merge({("l", "a", "g", "e", "r"): 2}, ("g", "e")) == {("l", "a", "ge", "r"): 2})
check("TODO 3b ohne Überlappung", lambda: merge({("a", "a", "a"): 1}, ("a", "a")) == {("aa", "a"): 1})
check("TODO 3c Häufigkeit bleibt", lambda: sum(merge(KORPUS, ("g", "e")).values()) == sum(KORPUS.values()))

Jetzt das eigentliche Training: acht Merge-Schritte.

In [ ]:
korpus = dict(KORPUS)
gelernte_merges = []
for schritt in range(8):
    paare = zaehle_paare(korpus)
    if not paare:
        print("⏳ Erst TODO 2 und 3 lösen."); break
    bestes, anzahl = paare.most_common(1)[0]
    korpus = merge(korpus, bestes)
    gelernte_merges.append(bestes)
    print(f"Schritt {schritt + 1}: {bestes[0]} + {bestes[1]} → {bestes[0] + bestes[1]}  ({anzahl}-mal)")

print("\nZerlegung nach dem Training:")
for wort in korpus:
    print("  " + " | ".join(wort))

> **Zum Nachdenken:** Nach wenigen Schritten ist „lager“ ein eigenes Token, und „kugellager“ wie „spindellager“ enden darauf. Ein unbekanntes Wort wie „wälzlager“ ließe sich trotzdem zerlegen. Warum kann BPE nie an einem unbekannten Wort scheitern?

## A3 · Wie zählen die Modelle der KI-Toolbox?

Jedes Modell bringt seinen eigenen Tokenizer mit. Mistral, DeepSeek und GLM wurden von verschiedenen Teams mit verschiedenen Daten trainiert und zerlegen denselben Text deshalb unterschiedlich. Die KI-Toolbox hat keinen eigenen Zähl-Endpunkt, aber jede Antwort meldet in `usage.prompt_tokens`, wie viele Tokens die Eingabe hatte. Ein Aufruf mit `max_tokens=1` genügt also zum Zählen.

In [ ]:
try:
    from google.colab import userdata
    API_KEY = userdata.get("KIT_TOOLBOX_KEY")
except ImportError:
    import os
    API_KEY = os.environ.get("KIT_TOOLBOX_KEY")
except Exception as e:
    API_KEY = None
    print("⚠️ Secret nicht lesbar:", type(e).__name__)
API_KEY = API_KEY.strip() if API_KEY else None
BASE_URL = "https://ki-toolbox.scc.kit.edu/api/v1"   # KI-Toolbox des KIT
client = OpenAI(api_key=API_KEY or "fehlt", base_url=BASE_URL, max_retries=3)
print("Schlüssel geladen:", bool(API_KEY))

### TODO 4 · Tokens mit den Modellen der KI-Toolbox zählen
Schreibt `modell_tokens(text, modell)`. Sie ruft `client.chat.completions.create(model=..., max_tokens=1, messages=[...])` mit einer einzigen Nutzernachricht auf und gibt `.usage.prompt_tokens` zurück.
Die Zahl enthält ein paar Tokens Verpackung für die Nachricht (das Chat-Format des Modells). Für den Vergleich ist das egal.

In [ ]:
def modell_tokens(text, modell="kit.mistral-small-4-119b-a8b"):
    # TODO 4
    pass

MODELLE_TOOLBOX = ['kit.mistral-small-4-119b-a8b', 'kit.deepseek-v4-flash', 'kit.glm-5.3-flash']
for modell in MODELLE_TOOLBOX:
    if modell_tokens("Test", modell) is None:
        print("⏳ Erst TODO 4 lösen."); break
    print(f"{modell:30s} Englisch: {modell_tokens(satz_en, modell):3d}  Deutsch: {modell_tokens(satz_de, modell):3d}")

In [ ]:
check("TODO 4 Zählung liefert eine positive ganze Zahl", lambda: isinstance(modell_tokens(satz_de), int) and modell_tokens(satz_de) > 0)

---
# Teil B · Von der Token-ID zum Vektor

Ein Modell hat eine **Embedding-Matrix** $E \in \mathbb{R}^{V \times d}$: eine Zeile pro Token, $d$ Zahlen pro Zeile. Bei GPT-2 ist $V = 50.257$ und $d = 768$.
Mathematisch wählt man die Zeile mit einem **One-Hot-Vektor** $e_i$ aus (eine 1 an Position $i$, sonst 0):

$$e_i^\top E = E_{i,:}$$

In der Praxis schreibt man einfach `E[i]`. Die Zahlen in $E$ sind Parameter, die beim Training gelernt werden (Woche 3).

### TODO 5 · One-Hot mal Matrix

In [ ]:
rng = np.random.default_rng(0)
V, d = 10, 4
E = rng.normal(size=(V, d))          # Mini-Embedding-Matrix: 10 Tokens, je 4 Zahlen

def one_hot(i, n):
    # TODO 5a: Vektor der Länge n mit einer 1 an Position i
    pass

# TODO 5b: Zeile 3 über die Matrixmultiplikation auswählen
zeile_3 = None
print(zeile_3, "\n", E[3])

In [ ]:
check("TODO 5a one_hot", lambda: one_hot(2, 5).tolist() == [0, 0, 1, 0, 0])
check("TODO 5b Zeilenauswahl", lambda: np.allclose(zeile_3, E[3]))

---
# Teil C · Satz-Embeddings und semantische Suche

Für ganze Sätze nutzen wir ein **Satz-Embedding-Modell**: Es macht aus jedem Text einen Vektor, und Texte mit ähnlicher Bedeutung bekommen ähnliche Richtungen.
Unser Modell `paraphrase-multilingual-MiniLM-L12-v2` ist mehrsprachig und klein genug für die CPU. Der erste Download dauert etwa eine Minute.

In [ ]:
modell = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
print("Dimension der Embeddings:", modell.get_embedding_dimension())
print("Tokenizer dieses Modells:", modell.tokenizer.tokenize("Spindelüberhitzung"))

Dieses Modell hat einen anderen, mehrsprachigen Tokenizer. Vergleicht die Zerlegung von „Spindelüberhitzung“ mit GPT-2 oben.

## C1 · 30 Störungsmeldungen einbetten

In [ ]:
MELDUNGEN = [
    ("mechanisch", "Spindel läuft unrund, deutliche Vibrationen beim Fräsen"),
    ("mechanisch", "Lager der Hauptspindel macht laute Mahlgeräusche"),
    ("mechanisch", "Werkzeugwechsler klemmt beim Greifen des Werkzeugs"),
    ("mechanisch", "Riemen am Antrieb der Förderstrecke gerissen"),
    ("mechanisch", "Schleifscheibe gebrochen, Maschine sofort gestoppt"),
    ("mechanisch", "Spänestau im Förderer blockiert den Austrag"),
    ("mechanisch", "Kugelgewindetrieb der X-Achse hat spürbares Spiel"),
    ("mechanisch", "Unwucht an der Welle, starke Schwingungen im Leerlauf"),
    ("elektrisch", "Not-Aus ausgelöst durch defekten Türschalter"),
    ("elektrisch", "Achsantrieb Y meldet Überstrom und schaltet ab"),
    ("elektrisch", "Sicherung im Schaltschrank ausgelöst"),
    ("elektrisch", "Sensor am Spannfutter liefert kein Signal mehr"),
    ("elektrisch", "Motorschutzschalter fällt beim Anlauf immer wieder raus"),
    ("elektrisch", "Kabelbruch an der Schleppkette der Z-Achse"),
    ("elektrisch", "Schweißstromquelle ausgefallen, keine Spannung am Ausgang"),
    ("elektrisch", "Lichtschranke an der Zuführung reagiert nicht"),
    ("Hydraulik/Kühlung", "Kühlmittelpumpe undicht, Pfütze unter der Maschine"),
    ("Hydraulik/Kühlung", "Hydraulikdruck fällt während des Spannvorgangs ab"),
    ("Hydraulik/Kühlung", "Kühlschmierstoff-Konzentration zu niedrig, Werkstücke rosten"),
    ("Hydraulik/Kühlung", "Ölleck am Hydraulikaggregat entdeckt"),
    ("Hydraulik/Kühlung", "Spindeltemperatur zu hoch, Kühlung arbeitet nicht richtig"),
    ("Hydraulik/Kühlung", "Filter im Kühlkreislauf verstopft, Durchfluss zu gering"),
    ("Hydraulik/Kühlung", "Druckluft schwankt, Spannzylinder schließen unzuverlässig"),
    ("Steuerung/Software", "CNC-Programm nach Update nicht mehr ladbar"),
    ("Steuerung/Software", "Steuerung hängt beim Hochfahren im Bootvorgang"),
    ("Steuerung/Software", "Werkzeugdaten nach Neustart verloren gegangen"),
    ("Steuerung/Software", "Messprotokoll wird nicht auf dem Server gespeichert"),
    ("Steuerung/Software", "Fehlermeldung 'Achsparameter ungültig' nach Softwarewechsel"),
    ("Steuerung/Software", "Verbindung zwischen SPS und Leitrechner bricht ab"),
    ("Steuerung/Software", "Prüfsoftware stürzt beim Messlauf ab"),
]
KATEGORIEN = [k for k, _ in MELDUNGEN]
TEXTE = [t for _, t in MELDUNGEN]
print(len(TEXTE), "Meldungen in", len(set(KATEGORIEN)), "Kategorien")

### TODO 6 · Einbetten und Ähnlichkeitsmatrix
1. Bettet alle `TEXTE` mit `modell.encode(..., normalize_embeddings=True)` ein. Die Vektoren haben dann Länge 1.
2. Berechnet die Matrix `S` aller paarweisen Kosinus-Ähnlichkeiten. Für Einheitsvektoren ist der Kosinus einfach das Skalarprodukt, also ist `S` eine einzige Matrixmultiplikation (Woche 2).

In [ ]:
# TODO 6
emb = None
S = None

print(None if emb is None else emb.shape, None if S is None else S.shape)

In [ ]:
check("TODO 6a Embeddings haben Länge 1", lambda: np.allclose(np.linalg.norm(emb, axis=1), 1, atol=1e-4))
check("TODO 6b S ist 30 × 30 und symmetrisch", lambda: S.shape == (30, 30) and np.allclose(S, S.T, atol=1e-5))
check("TODO 6c Diagonale ist 1", lambda: np.allclose(np.diag(S), 1, atol=1e-4))

In [ ]:
if S is not None:
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    im = ax.imshow(S, cmap="viridis")
    grenzen = [i for i in range(1, 30) if KATEGORIEN[i] != KATEGORIEN[i - 1]]
    for g in grenzen:
        ax.axhline(g - 0.5, color="white", lw=1); ax.axvline(g - 0.5, color="white", lw=1)
    ax.set_title("Kosinus-Ähnlichkeit, sortiert nach Kategorie"); plt.colorbar(im)
    plt.show()

    naechster = []
    for i in range(30):
        zeile = S[i].copy(); zeile[i] = -np.inf          # sich selbst ausschließen
        naechster.append(int(np.argmax(zeile)))
    gleich = np.mean([KATEGORIEN[i] == KATEGORIEN[j] for i, j in enumerate(naechster)])
    print(f"Nächster Nachbar hat dieselbe Kategorie: {gleich:.0%} (zufällig wären es etwa 23 %)")
    for i in [0, 9, 20, 23]:
        print(f"  {TEXTE[i]}\n    → {TEXTE[naechster[i]]}  ({S[i, naechster[i]]:.2f})")

> **Zum Nachdenken:** Wo liegt Meldung 21 („Spindeltemperatur zu hoch, Kühlung arbeitet nicht richtig“)? Bei den Kühlungs-Meldungen oder bei der Spindel? Was sagt das über Embeddings?

## C2 · Semantische Suche

Eine **semantische Suche** bettet die Suchanfrage mit demselben Modell ein und sucht die Meldungen mit der größten Kosinus-Ähnlichkeit. Genau das macht ab Woche 7 ein RAG-System.

### TODO 7 · Die drei passendsten Meldungen

In [ ]:
anfrage = "Die Hauptspindel vibriert und macht komische Geräusche"

# TODO 7: Anfrage einbetten (normalisiert), Ähnlichkeit zu allen Meldungen, Indizes der Top 3
scores = None
top3 = None

if top3 is not None:
    for i in top3:
        print(f"{scores[i]:.3f}  [{KATEGORIEN[i]}] {TEXTE[i]}")

In [ ]:
check("TODO 7a 30 Scores", lambda: scores.shape == (30,))
check("TODO 7b Top 3 absteigend", lambda: len(top3) == 3 and scores[top3[0]] >= scores[top3[1]] >= scores[top3[2]] and scores[top3[0]] == scores.max())

Probiert eigene Anfragen aus, auch auf Englisch. Das Modell ist mehrsprachig.

In [ ]:
if S is not None:
    for frage in ["Maschine wird zu heiß", "the machine lost power", "Software crasht"]:
        sc = emb @ modell.encode([frage], normalize_embeddings=True)[0]
        i = int(np.argmax(sc))
        print(f"{frage!r:28s} → {TEXTE[i]}  ({sc[i]:.2f})")

---
# Teil D · PCA: 384 Dimensionen sichtbar machen

Die Embeddings haben mehrere hundert Dimensionen. Die **Hauptkomponentenanalyse (PCA)** sucht die Richtungen, in denen die Daten am stärksten streuen, und projiziert darauf:

1. Daten zentrieren: $\tilde X = X - \bar x$
2. Kovarianzmatrix $C = \frac{1}{n}\tilde X^\top \tilde X$
3. Eigenvektoren von $C$ zu den zwei größten Eigenwerten: $V_2 = [v_1 \; v_2]$
4. Projektion $Z = \tilde X V_2$, eine Zeile pro Meldung mit zwei Koordinaten

Der Anteil $\lambda_1 + \lambda_2$ an $\sum_k \lambda_k$ sagt, wie viel der Streuung die Grafik zeigt.

### TODO 8 · PCA in NumPy
Tipp: `np.linalg.eigh(C)` liefert Eigenwerte **aufsteigend** sortiert und die Eigenvektoren als **Spalten**.

In [ ]:
def pca_2d(X):
    # TODO 8: Schritte 1 bis 4; zurückgeben: Z (n × 2) und den erklärten Anteil der Varianz (Zahl zwischen 0 und 1)
    Z, anteil = None, None
    return Z, anteil

In [ ]:
# Test mit Zufallsdaten gegen die Singulärwertzerlegung (Vorzeichen der Achsen sind beliebig)
Xt = np.random.default_rng(1).normal(size=(50, 6)) @ np.diag([5, 3, 1, 1, 0.5, 0.2])
Xtz = Xt - Xt.mean(axis=0)
U, sv, Vt = np.linalg.svd(Xtz, full_matrices=False)
check("TODO 8a Projektion stimmt (bis aufs Vorzeichen)", lambda: np.allclose(np.abs(pca_2d(Xt)[0]), np.abs(Xtz @ Vt[:2].T)))
check("TODO 8b erklärter Anteil", lambda: math.isclose(pca_2d(Xt)[1], (sv[:2] ** 2).sum() / (sv ** 2).sum()))

In [ ]:
if emb is not None and pca_2d(emb)[0] is not None:
    Z, anteil = pca_2d(emb)
    farben = {"mechanisch": "tab:blue", "elektrisch": "tab:orange", "Hydraulik/Kühlung": "tab:green", "Steuerung/Software": "tab:red"}
    plt.figure(figsize=(8, 6))
    for kat, farbe in farben.items():
        idx = [i for i, k in enumerate(KATEGORIEN) if k == kat]
        plt.scatter(Z[idx, 0], Z[idx, 1], c=farbe, label=kat, s=60)
    for i in range(len(TEXTE)):
        plt.annotate(str(i), (Z[i, 0], Z[i, 1]), fontsize=8, xytext=(3, 3), textcoords="offset points")
    plt.legend(); plt.title(f"PCA der Satz-Embeddings ({anteil:.0%} der Varianz sichtbar)")
    plt.show()

> **Vorsicht beim Deuten:** Die Grafik zeigt nur einen Teil der Streuung. Zwei Punkte, die in 2D nah beieinander liegen, können im vollen Raum weit auseinander sein. Für Entscheidungen zählt die Kosinus-Ähnlichkeit im Originalraum, nicht das Bild.

---
## Bonus

**B1 · Das Lager-Problem.** Berechnet die Kosinus-Ähnlichkeiten zwischen diesen drei Sätzen. Welche zwei sind sich am ähnlichsten, obwohl A und B das Wort „Lager“ teilen?
- A: „Das Lager der Spindel ist heiß gelaufen.“
- B: „Das Lager ist bis unter die Decke voll.“
- C: „Das Wälzlager der Antriebswelle muss getauscht werden.“

**B2 · Grenzen.** Vergleicht „Spindel läuft unrund“ mit „Spindel läuft nicht unrund“ sowie „Störung an M-03“ mit „Störung an M-05“. Was fällt euch auf?

**B3 · Embeddings aus der KI-Toolbox.** Die Toolbox bietet mit `kit.qwen3-embedding-8b` auch ein großes Embedding-Modell (8 Milliarden Parameter). Holt euch mit `client.embeddings.create(model=..., input=[...])` die Vektoren für die drei Sätze aus B1. Welche Dimension haben sie, und ändert sich die Reihenfolge der Ähnlichkeiten?

In [ ]:
# B1 / B2

## Zusammenfassung

- **Tokenizer** zerlegen Text in Stücke aus einem festen Wortschatz. **BPE** lernt diesen Wortschatz, indem es häufige Paare verschmilzt, und kann jedes Wort darstellen.
- Die **Tokenzahl** hängt von Sprache und Tokenizer ab. Sie bestimmt Rechenzeit, Platz im Kontextfenster und bei kommerziellen Anbietern die Kosten.
- Eine **Embedding-Matrix** ordnet jeder Token-ID einen gelernten Vektor zu. Mathematisch ist das One-Hot mal Matrix.
- **Satz-Embeddings** bilden ganze Texte auf Vektoren ab. Die **Kosinus-Ähnlichkeit** misst Bedeutungsnähe und ermöglicht semantische Suche.
- **PCA** projiziert auf die Richtungen größter Streuung, zeigt aber nur einen Teil des Bildes.

### Ausblick auf Woche 5
Ein statisches Token-Embedding für „Lager“ ist immer gleich, egal ob Kugellager oder Hochregallager. Wie bekommt ein Sprachmodell **Kontext** in seine Vektoren? Die Antwort heißt **Attention**.